FASE 01 — Instalação

In [1]:
#%pip install -U \
#    "langchain==1.3.14" \
#    langchain-community \
#    langchain-ollama \
#    langchain-text-splitters \
#    "psycopg[binary]" \
#    pgvector \
#    pymupdf \
#    ollama \
#    gradio \
#    gtts \
#    numpy \
#    pandas \
#    tqdm

In [2]:
import sys

from sympy.physics.units import temperature

# Check if running in Google Colab
if 'google.colab' in sys.modules:
    %pip install SpeechRecognition
    %pip install PyAudio
else:
    # For local environments, ensure PyAudio is installed manually if needed
    # or use conda/pip as appropriate for your setup.
    # SpeechRecognition generally installs fine, PyAudio often needs system-level dependencies.
    %pip install SpeechRecognition
    %pip install PyAudio

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Note: you may need to restart the kernel to use updated packages.


In [3]:
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\FABIO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

FASE 02 — Imports

In [4]:
import os
import regex as re
import json
import hashlib
import time

from pathlib import Path
from datetime import datetime

import fitz

import psycopg
import pandas as pd

import ollama
import gradio as gr

from gtts import gTTS

from langchain_ollama import OllamaEmbeddings

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

In [5]:
import speech_recognition as sr

FASE 03 — Configuração da aplicação

Altere principalmente PDF_DIRECTORY e as informações do PostgreSQL.

In [6]:
# ============================================================
# DIRETÓRIO DOS PDFS
# ============================================================

PDF_DIRECTORY = r"e:/meus_pdfs"


# ============================================================
# POSTGRESQL 18
# ============================================================

DB_HOST = "localhost"

DB_PORT = 5432

DB_NAME = "llm_sales_db"

DB_USER = "postgres"

DB_PASSWORD = "********"


# ============================================================
# OLLAMA
# ============================================================

OLLAMA_HOST = "http://localhost:11434"


# ============================================================
# MODELOS
# ============================================================

# Modelo para embeddings
EMBEDDING_MODEL = "nomic-embed-text"

# nomic-embed-text gera 768 dimensões
EMBEDDING_DIM = 768

# Modelo para responder perguntas
LLM_MODEL = "qwen3:8b"

TEMPERATURE = 0.1




# ============================================================
# TABELA FÍSICA
# ============================================================

CONTENT_TABLE = "rag_conteudo_5"


# ============================================================
# RAG
# ============================================================

TOP_K = 8

# Configurações para chunking tradicional (ainda presente, mas não usado com semantic chunking)
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# Configurações para chunking semântico
SEMANTIC_CHUNK_THRESHOLD = 0.8  # Limite de similaridade para quebrar chunks
MIN_SEMANTIC_CHUNK_SIZE = 200   # Tamanho mínimo de caracteres para um chunk semântico
MAX_SEMANTIC_CHUNK_SIZE = 1500  # Tamanho máximo de caracteres para um chunk semântico


# ============================================================
# ÁUDIO
# ============================================================

AUDIO_DIRECTORY = Path(
    "e:/audio"
)

AUDIO_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# INTERFACE
# ============================================================

GERAR_AUDIO_PADRAO = False


# ============================================================
# CONNECTION STRING
# ============================================================

CONNECTION_STRING = (
    f"postgresql://"
    f"{DB_USER}:"
    f"{DB_PASSWORD}@"
    f"{DB_HOST}:"
    f"{DB_PORT}/"
    f"{DB_NAME}"
)

print("=" * 70)


FASE 04 — Conexão com PostgreSQL

In [7]:
def get_db_connection():

    return psycopg.connect(

        host=DB_HOST,

        port=int(DB_PORT),

        dbname=DB_NAME,

        user=DB_USER,

        password=DB_PASSWORD,

        row_factory=psycopg.rows.dict_row

    )

Teste:

In [8]:
with get_db_connection() as conn:

    with conn.cursor() as cur:

        cur.execute(
            "SELECT version();"
        )

        resultado = cur.fetchone()


print(
    resultado["version"]
)

PostgreSQL 18.1 on x86_64-windows, compiled by msvc-19.44.35221, 64-bit


FASE 05 — Ativar pgvector

In [9]:
with get_db_connection() as conn:

    with conn.cursor() as cur:

        cur.execute(
            "CREATE EXTENSION IF NOT EXISTS vector;"
        )

    conn.commit()


print(
    "Extensão pgvector habilitada."
)

Extensão pgvector habilitada.


FASE 06 — Criar a tabela física

Agora a tabela contém somente texto e metadados textuais.

In [10]:
SQL_CREATE_TABLE = f"""

CREATE TABLE IF NOT EXISTS {CONTENT_TABLE} (

    id BIGSERIAL PRIMARY KEY,

    documento_hash VARCHAR(64) NOT NULL,

    nome_arquivo TEXT NOT NULL,

    caminho_arquivo TEXT,

    pagina INTEGER,

    chunk_numero INTEGER,

    tipo_conteudo VARCHAR(20)
        NOT NULL
        DEFAULT 'texto',

    content TEXT NOT NULL,

    embedding VECTOR({EMBEDDING_DIM}) NOT NULL,

    metadata JSONB,

    data_ingestao TIMESTAMPTZ
        DEFAULT NOW(),

    CONSTRAINT chk_tipo_conteudo

        CHECK (
            tipo_conteudo = 'texto'
        )

);

"""


with get_db_connection() as conn:

    with conn.cursor() as cur:

        cur.execute(
            SQL_CREATE_TABLE
        )

    conn.commit()


print(
    f"Tabela {CONTENT_TABLE} criada."
)

Tabela rag_conteudo_5 criada.


FASE 07 — Criar índices

In [11]:
with get_db_connection() as conn:

    with conn.cursor() as cur:

        # Índice vetorial
        cur.execute(f"""

            CREATE INDEX IF NOT EXISTS
            idx_rag_conteudo_embedding

            ON {CONTENT_TABLE}

            USING hnsw (

                embedding
                vector_cosine_ops

            );

        """)


        # Índice por documento
        cur.execute(f"""

            CREATE INDEX IF NOT EXISTS
            idx_rag_conteudo_documento

            ON {CONTENT_TABLE} (

                documento_hash

            );

        """)


        # Índice por arquivo
        cur.execute(f"""

            CREATE INDEX IF NOT EXISTS
            idx_rag_conteudo_arquivo

            ON {CONTENT_TABLE} (

                nome_arquivo

            );

        """)


        # Índice por página
        cur.execute(f"""

            CREATE INDEX IF NOT EXISTS
            idx_rag_conteudo_pagina

            ON {CONTENT_TABLE} (

                pagina

            );

        """)


    conn.commit()


print(
    "Índices criados."
)

Índices criados.


FASE 08 — Configurar Ollama

In [12]:
os.environ["OLLAMA_HOST"] = OLLAMA_HOST

print(
    "OLLAMA_HOST =",
    OLLAMA_HOST
)

OLLAMA_HOST = http://localhost:11434


FASE 09 — Verificar modelos Ollama

In [13]:
modelos = ollama.list()

print(
    "MODELOS OLLAMA DISPONÍVEIS"
)

print("=" * 60)

for modelo in modelos["models"]:

    print(
        modelo["model"]
    )

MODELOS OLLAMA DISPONÍVEIS
qwen3:8b
nomic-embed-text:latest
qwen2.5vl:7b
llama3.1:latest
llava:latest
all-minilm:latest
llama3:latest
granite3.3:latest
llama2:latest
glm-5:cloud


Caso ainda não estejam instalados:

In [14]:
ollama.pull(
    "nomic-embed-text"
)

ollama.pull(
    "qwen3:8b"
)

ProgressResponse(status='success', completed=None, total=None, digest=None)

FASE 10 — Configurar embeddings

In [15]:
embeddings = OllamaEmbeddings(

    model=EMBEDDING_MODEL

#    base_url=OLLAMA_HOST

)


teste_embedding = embeddings.embed_query(

    "Teste de embedding PostgreSQL."

)


print(
    "Dimensão do vetor:",
    len(teste_embedding)
)


if len(teste_embedding) != EMBEDDING_DIM:

    raise ValueError(

        f"Dimensão incorreta. "
        f"Esperado {EMBEDDING_DIM}, "
        f"recebido {len(teste_embedding)}."

    )


print(
    "Embedding validado com sucesso."
)

Dimensão do vetor: 768
Embedding validado com sucesso.


FASE 11 — Configurar o Text Splitter

In [16]:
splitter = RecursiveCharacterTextSplitter(

    chunk_size=CHUNK_SIZE,

    chunk_overlap=CHUNK_OVERLAP,

    separators=[

        "\n\n",

        "\n",

        ". ",

        " ",

        "",
        "_",
        "*",
        "#"

    ]

)


print(
    "Text splitter configurado."
)

Text splitter configurado.


In [17]:
import numpy as np

def semantic_chunk_text(
    text,
    embeddings_model,
    threshold=SEMANTIC_CHUNK_THRESHOLD,
    min_chunk_size_chars=MIN_SEMANTIC_CHUNK_SIZE,
    max_chunk_size_chars=MAX_SEMANTIC_CHUNK_SIZE
):
    sentences = nltk.sent_tokenize(text, language='portuguese')
    if not sentences:
        return []

    # Get embeddings for all sentences
    sentence_embeddings = embeddings_model.embed_documents(sentences)

    chunks = []
    current_chunk_sentences = []
    current_chunk_chars = 0

    for i in range(len(sentences)):
        sentence = sentences[i]
        sentence_embedding = sentence_embeddings[i]

        if not current_chunk_sentences:
            # Always add the first sentence to the current chunk
            current_chunk_sentences.append(sentence)
            current_chunk_chars += len(sentence)
        else:
            # Calculate similarity with the *previous* sentence's embedding
            prev_sentence_embedding = sentence_embeddings[i-1]

            # Handle potential zero norm for embeddings (though unlikely with good models)
            norm_prod = np.linalg.norm(sentence_embedding) * np.linalg.norm(prev_sentence_embedding)
            if norm_prod == 0:
                similarity = 0.0
            else:
                similarity = np.dot(sentence_embedding, prev_sentence_embedding) / norm_prod

            # Decide whether to split or continue the chunk
            if similarity < threshold and current_chunk_chars >= min_chunk_size_chars:
                # If similarity is low and current chunk is large enough, finalize current and start a new one
                chunks.append(" ".join(current_chunk_sentences))
                current_chunk_sentences = [sentence]
                current_chunk_chars = len(sentence)
            elif current_chunk_chars + len(sentence) > max_chunk_size_chars and current_chunk_chars > 0:
                # If adding the next sentence exceeds max chunk size, finalize current and start new
                chunks.append(" ".join(current_chunk_sentences))
                current_chunk_sentences = [sentence]
                current_chunk_chars = len(sentence)
            else:
                # Otherwise, continue the current chunk
                current_chunk_sentences.append(sentence)
                current_chunk_chars += len(sentence)

    # Add the last chunk if it's not empty
    if current_chunk_sentences:
        chunks.append(" ".join(current_chunk_sentences))

    return chunks


FASE 12 — Calcular hash do PDF

O hash evita inserir o mesmo documento duas vezes.

FASE 11.5 — Chunking Semântico

O chunking semântico é uma técnica que agrupa texto em pedaços (chunks) com base na similaridade de significado entre as frases, em vez de simplesmente cortar o texto por um número fixo de caracteres. Isso ajuda a manter o contexto e a coerência dentro de cada chunk, o que pode melhorar a qualidade das recuperações em sistemas RAG.

Nesta fase, iremos:

1.  **Dividir o texto em sentenças:** Usaremos o `nltk` para tokenizar o texto de cada página em sentenças.
2.  **Gerar embeddings para cada sentença:** Utilizaremos o modelo de embeddings já configurado para criar representações vetoriais de cada sentença.
3.  **Calcular similaridade:** Avaliaremos a similaridade de cosseno entre sentenças adjacentes.
4.  **Criar chunks:** Quando a similaridade entre sentenças cai abaixo de um certo `threshold` (limite), consideramos isso um ponto de quebra semântico, iniciando um novo chunk. Também garantiremos que os chunks tenham um tamanho mínimo e máximo para otimizar o desempenho do RAG.

In [18]:
def calcular_hash(
    caminho
):

    sha256 = hashlib.sha256()


    with open(
        caminho,
        "rb"
    ) as arquivo:

        while True:

            bloco = arquivo.read(
                1024 * 1024
            )

            if not bloco:

                break

            sha256.update(
                bloco
            )


    return sha256.hexdigest()

FASE 13 — Verificar se documento já existe

In [19]:
def documento_ja_processado(
    documento_hash
):

    sql = f"""

        SELECT COUNT(*) AS quantidade

        FROM {CONTENT_TABLE}

        WHERE documento_hash = %s;

    """


    with get_db_connection() as conn:

        with conn.cursor() as cur:

            cur.execute(

                sql,

                (
                    documento_hash,
                )

            )

            resultado = cur.fetchone()


    return (
        resultado["quantidade"] > 0
    )

FASE 14 — Extrair texto dos PDFs

Aqui não existe mais qualquer processamento de imagem.

In [20]:
def extrair_texto_pdf(
    caminho_pdf
):

    paginas = []


    doc = fitz.open(
        caminho_pdf
    )


    try:

        for numero_pagina, pagina in enumerate(

            doc,

            start=1

        ):

            texto = pagina.get_text(
                "text"
            )


            if texto and texto.strip():

                paginas.append({

                    "pagina":
                        numero_pagina,

                    "texto":
                        texto.strip()

                })


    finally:

        doc.close()


    return paginas

FASE 15 — Inserir texto no PostgreSQL

In [21]:
def inserir_texto(

    documento_hash,

    nome_arquivo,

    caminho_arquivo,

    pagina,

    chunk_numero,

    texto

):

    vetor = embeddings.embed_query(
        texto
    )


    if len(vetor) != EMBEDDING_DIM:

        raise ValueError(

            f"Embedding inválido. "
            f"Esperado {EMBEDDING_DIM}, "
            f"recebido {len(vetor)}."

        )


    metadata = {

        "tipo":
            "texto",

        "pagina":
            pagina,

        "arquivo":
            nome_arquivo

    }


    sql = f"""

    INSERT INTO {CONTENT_TABLE} (

        documento_hash,

        nome_arquivo,

        caminho_arquivo,

        pagina,

        chunk_numero,

        tipo_conteudo,

        content,

        embedding,

        metadata

    )

    VALUES (

        %s,
        %s,
        %s,
        %s,
        %s,
        'texto',
        %s,
        %s,
        %s

    );

    """


    with get_db_connection() as conn:

        with conn.cursor() as cur:

            cur.execute(

                sql,

                (

                    documento_hash,

                    nome_arquivo,

                    caminho_arquivo,

                    pagina,

                    chunk_numero,

                    texto,

                    vetor,

                    json.dumps(

                        metadata,

                        ensure_ascii=False

                    )

                )

            )

        conn.commit()

FASE 16 — Ingestão de um PDF

In [22]:
def ingerir_pdf(
    caminho_pdf
):

    caminho_pdf = Path(
        caminho_pdf
    )


    print()
    print("=" * 70)
    print(
        f"Processando: {caminho_pdf.name}"
    )
    print("=" * 70)


    # ========================================================
    # HASH
    # ========================================================

    documento_hash = calcular_hash(
        caminho_pdf
    )


    print(
        "Hash:",
        documento_hash
    )


    # ========================================================
    # VERIFICAR DUPLICIDADE
    # ========================================================

    if documento_ja_processado(
        documento_hash
    ):

        print(
            "Documento já existe no PostgreSQL."
        )

        return 0


    # ========================================================
    # EXTRAIR TEXTO
    # ========================================================

    paginas = extrair_texto_pdf(
        caminho_pdf
    )


    total_chunks = 0


    # ========================================================
    # PROCESSAR PÁGINAS COM CHUNKING SEMÂNTICO
    # ========================================================

    for pagina_info in paginas:

        pagina = pagina_info[
            "pagina"
        ]

        texto = pagina_info[
            "texto"
        ]


        # Usar o chunking semântico
        chunks_conteudo = semantic_chunk_text(
            text=texto,
            embeddings_model=embeddings,
            threshold=SEMANTIC_CHUNK_THRESHOLD,
            min_chunk_size_chars=MIN_SEMANTIC_CHUNK_SIZE,
            max_chunk_size_chars=MAX_SEMANTIC_CHUNK_SIZE
        )


        for chunk_text in chunks_conteudo:

            total_chunks += 1


            inserir_texto(

                documento_hash=
                    documento_hash,

                nome_arquivo=
                    caminho_pdf.name,

                caminho_arquivo=
                    str(caminho_pdf),

                pagina=
                    pagina,

                chunk_numero=
                    total_chunks,

                texto=
                    chunk_text

            )


    print(
        f"Chunks inseridos: {total_chunks}"
    )

    print(
        f"Total de páginas: {len(paginas)}"
    )


    print(
        "Documento processado."
    )


    return total_chunks

FASE 17 — Ingestão de múltiplos PDFs

O diretório continua sendo configurável pela variável:

In [23]:
PDF_DIRECTORY

'e:/meus_pdfs'

In [24]:
def ingerir_diretorio(
    diretorio
):

    diretorio = Path(
        diretorio
    )


    if not diretorio.exists():

        raise FileNotFoundError(

            f"Diretório não encontrado: "
            f"{diretorio}"

        )


    arquivos = sorted(

        diretorio.glob(
            "*.pdf"
        )

    )


    print("=" * 70)

    print(
        "DIRETÓRIO:",
        diretorio
    )

    print(
        "PDFs encontrados:",
        len(arquivos)
    )

    print("=" * 70)


    total = 0


    for indice, pdf in enumerate(

        arquivos,

        start=1

    ):

        print()

        print(
            f"[{indice}/{len(arquivos)}]"
        )


        try:

            total += ingerir_pdf(
                pdf
            )


        except Exception as erro:

            print(
                "ERRO:",
                pdf.name
            )

            print(
                repr(erro)
            )


    print()
    print("=" * 70)

    print(
        "INGESTÃO FINALIZADA"
    )

    print(
        "Total de chunks:",
        total
    )

    print("=" * 70)


    return total

FASE 18 — Executar ingestão

In [25]:
total_chunks = ingerir_diretorio(
    PDF_DIRECTORY
)

DIRETÓRIO: e:\meus_pdfs
PDFs encontrados: 91

[1/91]

Processando: 2026_Largelanguagemodelsnacinciadainformao.pdf
Hash: ee4992c8e7ed6453e7752a397d41e6a1d17a16e0304a2066a32e44e1a2acd3e3
Documento já existe no PostgreSQL.

[2/91]

Processando: 4394+.pdf
Hash: 92dd10f71766b52eff8c0ca05f0813618cd96dff77f9e006b8d9637ce53c2067
Documento já existe no PostgreSQL.

[3/91]

Processando: 5 Geração Aumentada por Recuperação (RAG) – Processamento de Linguagem Natural.pdf
Hash: f46fbc91f274eb2507f3570e5ea934fa7628400c2d6eb4718f324afc18aec100
Documento já existe no PostgreSQL.

[4/91]

Processando: Agentes de IA para Desenvolvimento de Projetos.pdf
Hash: b038547254840006737cf062a216058916bbbee2600021489d58768f687b9e82
Documento já existe no PostgreSQL.

[5/91]

Processando: Agentes_com_SLM_em_Produção_Arquitetura,_Escala_e_Aplicações_Reais.pdf
Hash: 0a3b00def823de0e062f99eba0fb4bbb7aad4cc15c269e202129f29a577e302f
Documento já existe no PostgreSQL.

[6/91]

Processando: Agentes_de_IA_Do_Conceito_à_Prá

FASE 19 — Verificar dados

In [26]:
sql = f"""

SELECT

    COUNT(*) AS total_registros,

    COUNT(
        DISTINCT documento_hash
    ) AS documentos,

    COUNT(
        DISTINCT nome_arquivo
    ) AS arquivos,

    COUNT(
        DISTINCT pagina
    ) AS paginas,

    COUNT(
        DISTINCT embedding
    ) AS embeddings

FROM {CONTENT_TABLE};

"""


with get_db_connection() as conn:

    df = pd.read_sql(
        sql,
        conn
    )


df

C:\Users\FABIO\AppData\Local\Temp\ipykernel_28064\1107770143.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,total_registros,documentos,arquivos,paginas,embeddings
0,total_registros,documentos,arquivos,paginas,embeddings


FASE 20 — Listar documentos

In [27]:
sql = f"""

SELECT

    nome_arquivo,

    COUNT(*) AS chunks,

    MIN(pagina) AS primeira_pagina,

    MAX(pagina) AS ultima_pagina,

    MIN(data_ingestao)
        AS data_ingestao

FROM {CONTENT_TABLE}

GROUP BY nome_arquivo

ORDER BY nome_arquivo;

"""


with get_db_connection() as conn:

    df_documentos = pd.read_sql(
        sql,
        conn
    )


df_documentos

C:\Users\FABIO\AppData\Local\Temp\ipykernel_28064\2049182090.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_documentos = pd.read_sql(


,nome_arquivo,chunks,primeira_pagina,ultima_pagina,data_ingestao
0,nome_arquivo,chunks,primeira_pagina,ultima_pagina,data_ingestao
1,nome_arquivo,chunks,primeira_pagina,ultima_pagina,data_ingestao
2,nome_arquivo,chunks,primeira_pagina,ultima_pagina,data_ingestao
3,nome_arquivo,chunks,primeira_pagina,ultima_pagina,data_ingestao
4,nome_arquivo,chunks,primeira_pagina,ultima_pagina,data_ingestao
...,...,...,...,...,...
80,nome_arquivo,chunks,primeira_pagina,ultima_pagina,data_ingestao
81,nome_arquivo,chunks,primeira_pagina,ultima_pagina,data_ingestao
82,nome_arquivo,chunks,primeira_pagina,ultima_pagina,data_ingestao
83,nome_arquivo,chunks,primeira_pagina,ultima_pagina,data_ingestao


FASE 21 — Visualizar alguns registros

In [28]:
sql = f"""

SELECT

    id,

    nome_arquivo,

    pagina,

    chunk_numero,

    LEFT(
        content,
        500
    ) AS conteudo,

    data_ingestao

FROM {CONTENT_TABLE}

ORDER BY id

LIMIT 20;

"""


with get_db_connection() as conn:

    df = pd.read_sql(
        sql,
        conn
    )


df

C:\Users\FABIO\AppData\Local\Temp\ipykernel_28064\3390275124.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
0,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
1,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
2,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
3,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
4,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
5,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
6,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
7,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
8,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
9,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao


FASE 22 — Busca vetorial

Esta é a consulta utilizada pelo RAG.

In [29]:
def buscar_conteudo(

    pergunta,

    top_k=TOP_K

):

    vetor_pergunta = (
        embeddings.embed_query(
            pergunta
        )
    )


    if len(vetor_pergunta) != EMBEDDING_DIM:

        raise ValueError(
            "Dimensão do embedding da "
            "pergunta inválida."
        )


    sql = f"""

    SELECT

        id,

        documento_hash,

        nome_arquivo,

        caminho_arquivo,

        pagina,

        chunk_numero,

        tipo_conteudo,

        content,

        metadata,

        1 - (

            embedding
            <=> %s::vector

        ) AS similaridade


    FROM {CONTENT_TABLE}


    ORDER BY

        embedding
        <=> %s::vector


    LIMIT %s;

    """


    with get_db_connection() as conn:

        with conn.cursor() as cur:

            cur.execute(

                sql,

                (

                    vetor_pergunta,

                    vetor_pergunta,

                    top_k

                )

            )

            resultados = cur.fetchall()


    return resultados

FASE 23 — Testar busca vetorial

In [30]:
pergunta_teste = (
    "Qual é o assunto principal "
    "dos documentos?"
)


resultados = buscar_conteudo(
    pergunta_teste
)


for item in resultados:

    print("=" * 70)

    print(
        "Arquivo:",
        item["nome_arquivo"]
    )

    print(
        "Página:",
        item["pagina"]
    )

    print(
        "Similaridade:",
        round(
            float(
                item["similaridade"]
            ),
            4
        )
    )

    print()

    print(
        item["content"][:1000]
    )

Arquivo: Fragmentação em RAG_ um guia para a arquitetura da IBM.pdf
Página: 9
Similaridade: 0.7336

–
Por documento: trate cada arquivo PDF como um documento individual para ﬁns de
indexação. –
Por página: como alternativa, divida os PDFs em partes com base em páginas ou
seções lógicas, se necessário.
Arquivo: NLP 25.2- Recuperação_Avançada_com_RAG.pdf
Página: 3
Similaridade: 0.7279

Sem Domínio Específico
Não acessa documentos privados ou informações 
internas não públicas. A Solução RAG
Conecta o LLM a uma base externa verificada, garantindo precisão, transparência e acesso a dados atualizados.
Arquivo: monografia.pdf
Página: 29
Similaridade: 0.7205

Com os documentos criados, podemos abordar agora o código principal da LLM. A complexidade não é elevada. Resumidamente, o programa main.py é um loop While
que fica aguardando alguma pergunta do usuário.
Arquivo: LLMS E VETORIZAÇÃO SEMÂNTICA PARA INTERAÇÃO MULTIMODAL COM DOCUMENTOS PDF_ UMA ABORDAGEM DE GERAÇÃO AUMENTADA POR RECUPERAÇ

FASE 24 — Construir contexto do RAG

In [31]:
def construir_contexto(
    resultados
):

    partes = []


    for indice, item in enumerate(

        resultados,

        start=1

    ):

        partes.append(

            f"""
[FONTE {indice}]

Arquivo:
{item["nome_arquivo"]}

Página:
{item["pagina"]}

Similaridade:
{float(item["similaridade"]):.4f}

Conteúdo:
{item["content"]}

"""

        )


    return "\n".join(
        partes
    )

FASE 25 — Gerar resposta com Qwen3:8B

In [32]:
def gerar_resposta(

    pergunta,

    contexto

):

    prompt = f"""

Você é um assistente especializado
em responder perguntas sobre
documentos.

Dê respostas diretas.

Responda em português.

Quando encontrar um termo em inglês pronuncie corretamente.

Utilize somente as informações
presentes no contexto fornecido.

Não invente informações.

Não inclua a fonte.

Não inclua a referência bibligrafica.

Não inclua o autor

Se a resposta não estiver no
contexto, responda:

"Não encontrei essa informação
nos documentos."

Quando encontrar valores exiba

Quando encontrar valores em moedas exiba

Seja claro, objetivo e técnico.


PERGUNTA:

{pergunta}


CONTEXTO:

{contexto}


RESPOSTA:

"""


    resposta = ollama.chat(

        model=LLM_MODEL,



        messages=[

            {

                "role": "user",

                "content": prompt

            }

        ]

    )


    return (
        resposta[
            "message"
        ][
            "content"
        ]
    )

FASE 26 — Gerar áudio

O áudio não é gerado durante a consulta vetorial.

Ele só será chamado depois de gerar_resposta() terminar.

In [33]:
def gerar_audio(texto):

    timestamp = (
        datetime.now()
        .strftime(
            "%Y%m%d_%H%M%S_%f"
        )
    )


    caminho = (

        AUDIO_DIRECTORY
        / f"resposta_{timestamp}.mp3"

    )


    tts = gTTS(

        text=texto,

        lang="pt"

    )


    tts.save(
        str(caminho)
    )


    return str(caminho)

FASE 27 — Pipeline RAG completo

Esta é a função principal.

In [34]:
def executar_rag(
    pergunta
):
    inicio = time.time()

    # ========================================================
    # 1. VALIDAR PERGUNTA
    # ========================================================

    if not pergunta:

        return (

            "Digite uma pergunta.",

            "",

            None

        )


    pergunta = pergunta.strip()


    if not pergunta:

        return (

            "Digite uma pergunta.",

            "",

            None

        )


    # ========================================================
    # 2. BUSCA NO POSTGRESQL
    # ========================================================

    print(
        "1/3 - Buscando documentos..."
    )


    resultados = buscar_conteudo(

        pergunta,

        TOP_K

    )


    if not resultados:

        return (

            "Nenhum conteúdo encontrado.",

            "",

            None

        )


    # ========================================================
    # 3. CONTEXTO
    # ========================================================

    print(
        "2/3 - Construindo contexto..."
    )


    contexto = construir_contexto(

        resultados

    )


    # ========================================================
    # 4. RESPOSTA ESCRITA
    # ========================================================

    print(
        "3/3 - Gerando resposta escrita..."
    )


    resposta = gerar_resposta(

        pergunta,

        contexto

    )


    # Audio is no longer automatically generated here.
    # It will be generated on demand by the 'gerar_audio_btn' in Gradio.


    # ========================================================
    # FONTES
    # ========================================================

    fontes = []


    for item in resultados:

        fontes.append({

            "id":
                item["id"],

            "arquivo":
                item["nome_arquivo"],

            "pagina":
                item["pagina"],

            "chunk":
                item["chunk_numero"],

            "similaridade":
                round(

                    float(
                        item["similaridade"]
                    ),

                    4

                )

        })


    fontes_json = json.dumps(

        fontes,

        ensure_ascii=False,

        indent=2

    )


    tempo = (
        time.time()
        - inicio
    )


    print(
        f"Tempo total: {tempo:.2f}s"
    )


    # Return None for audio_path as it's not generated automatically
    return (

        resposta,

        fontes_json,

        None

    )

In [35]:
def gerar_audio_da_resposta(resposta_textual):
    if resposta_textual and resposta_textual != "Nenhum conteúdo encontrado." and resposta_textual != "Digite uma pergunta.":
        print("Gerando áudio...")
        # Remover caracteres indesejados e substituir 'IA' por 'Inteligência Artificial'
        texto_limpo = re.sub(r'\*', ' ', resposta_textual).replace('#','').replace('&','')
        texto_limpo = re.sub(r'\bIA\b', 'Inteligência Artificial', texto_limpo, flags=re.IGNORECASE)

        audio_path = gerar_audio(texto_limpo)
        print("Áudio gerado em: ", audio_path)
        return audio_path
    return None

In [36]:
def transcrever_audio_e_executar_rag():
    r = sr.Recognizer()
    try:
        with sr.Microphone() as source:
            print("Ouvindo... Fale sua pergunta.")
            r.adjust_for_ambient_noise(source) # Adjust for ambient noise
            audio = r.listen(source, timeout=5) # Listen for up to 5 seconds
            print("Transcrevendo...")
            pergunta_audio = r.recognize_google(audio, language='pt-BR')
            print(f"Você disse: {pergunta_audio}")

            resposta, fontes, _ = executar_rag(pergunta_audio)
            # Audio is not generated here, return None for audio_output
            return pergunta_audio, resposta, fontes, None

    except sr.WaitTimeoutError:
        print("Nenhuma fala detectada.")
        return "Nenhuma fala detectada.", "Nenhum conteúdo encontrado.", "", None
    except sr.UnknownValueError:
        print("Não foi possível entender o áudio.")
        return "Não foi possível entender o áudio.", "Nenhum conteúdo encontrado.", "", None
    except sr.RequestError as e:
        print(f"Erro no serviço de reconhecimento de fala; {e}")
        return f"Erro no serviço de reconhecimento de fala; {e}", "Nenhum conteúdo encontrado.", "", None
    except Exception as e:
        print(f"Ocorreu um erro: {e}")
        return f"Ocorreu um erro: {e}", "Nenhum conteúdo encontrado.", "", None

In [37]:
def limpar_campos():
    return "", "", None, ""

FASE 28 — Testar RAG sem áudio

FASE 29 — Testar RAG com áudio

In [38]:
resposta, fontes, audio_path = executar_rag(

    "O QUE É RAG?"

)


print("=" * 70)

print("RESPOSTA")

print("=" * 70)

print(resposta)

print()

print(
    "Áudio: " + (audio_path if audio_path else "Não gerado ou erro")
)

print()

print(
    "FONTES:"
)

print(fontes)

1/3 - Buscando documentos...
2/3 - Construindo contexto...
3/3 - Gerando resposta escrita...
Tempo total: 103.80s
RESPOSTA
RAG é uma técnica chamada **Retrieval-Augmented Generation**. Ela permite que um modelo de linguagem (LLM) acesse, recupere e utilize informações de uma base de dados externa antes de gerar uma resposta, adicionando conhecimento atualizado e contextualizando a geração de respostas. Essa abordagem resolve o problema do conhecimento estático dos modelos pré-treinados, integrando dados dinâmicos durante a inferência.

Áudio: Não gerado ou erro

FONTES:
[
  {
    "id": 6076,
    "arquivo": "Revisão_Completa_NLP,_LLM,_RAG_e_Agentes_de_IA .pdf",
    "pagina": 70,
    "chunk": 138,
    "similaridade": 0.7937
  },
  {
    "id": 9027,
    "arquivo": "RAG_doc.pdf",
    "pagina": 2,
    "chunk": 9,
    "similaridade": 0.7795
  },
  {
    "id": 5257,
    "arquivo": "RAG .pdf",
    "pagina": 29,
    "chunk": 52,
    "similaridade": 0.754
  },
  {
    "id": 3729,
    "arquivo": 

Pergunta
   │
   ▼
Embedding
   │
   ▼
PostgreSQL + pgvector
   │
   ▼
Contexto
   │
   ▼
Qwen3:8b
   │
   ▼
RESPOSTA ESCRITA
   │
   ├───────────────► retorna imediatamente
   │
   ▼
Se solicitado
   │
   ▼
gTTS
   │
   ▼
MP3

FASE 30 — Interface Gradio

In [39]:
with gr.Blocks(

    title="RAG PostgreSQL"

) as demo:

    # ========================================================
    # CABEÇALHO
    # ========================================================

    gr.Markdown(
        """
        # 📚 RAG com PostgreSQL + pgvector

        Sistema de perguntas e respostas
        sobre documentos PDF.

        **PostgreSQL 18 + pgvector + Ollama + Qwen3:8B**
        """
    )


    # ========================================================
    # PERGUNTA
    # ========================================================

    pergunta = gr.Textbox(

        label="Pergunta",

        placeholder=(

            "Digite sua pergunta "
            "sobre os documentos..."

        ),

        lines=4,

        interactive=True

    )


    # ========================================================
    # BOTÕES DE AÇÃO
    # ========================================================

    with gr.Row():

        consultar = gr.Button(

            "🔎 Consultar",

            variant="primary"

        )

        gerar_audio_btn = gr.Button(

            "🔊 Gerar Áudio"

        )

        limpar_btn = gr.Button(

            "🗑️ Limpar"

        )


    # ========================================================
    # RESPOSTA
    # ========================================================

    gr.Markdown(
        "## 📝 Resposta"
    )


    resposta = gr.Markdown()


    # ========================================================
    # ÁUDIO
    # ========================================================

    gr.Markdown(
        "## 🔊 Áudio"
    )


    audio_output = gr.Audio(

        label="Resposta em áudio",

        type="filepath",

        autoplay=False

    )


    # ========================================================
    # FONTES
    # ========================================================

    gr.Markdown(
        "## 📚 Fontes"
    )


    fontes = gr.Code(

        label="Documentos utilizados",

        language="json"

    )


    # ========================================================
    # EVENTOS
    # ========================================================


    consultar.click(

        fn=executar_rag,

        inputs=[pergunta],

        outputs=[

            resposta,

            fontes

        ]

    )


    pergunta.submit(

        fn=executar_rag,

        inputs=[pergunta],

        outputs=[

            resposta,

            fontes

        ]

    )


    gerar_audio_btn.click(

        fn=gerar_audio_da_resposta,

        inputs=[resposta],

        outputs=[audio_output]

    )


    limpar_btn.click(

        fn=limpar_campos,

        inputs=[],

        outputs=[

            pergunta,

            resposta,

            audio_output,

            fontes

        ]

    )

In [40]:
with gr.Blocks(
    title="RAG PostgreSQL"
) as demo:

    # ========================================================
    # CABEÇALHO
    # ========================================================

    gr.Markdown(
        """
        # 📚 RAG com PostgreSQL + pgvector

        Sistema de perguntas e respostas
        sobre documentos PDF.

        **PostgreSQL 18 + pgvector + Ollama + Qwen3:8B**
        """
    )


    # ========================================================
    # PERGUNTA
    # ========================================================

    pergunta = gr.Textbox(

        label="Pergunta",

        placeholder=(

            "Digite sua pergunta "
            "sobre os documentos..."

        ),

        lines=4,

        interactive=True

    )


    # ========================================================
    # BOTÕES DE AÇÃO
    # ========================================================

    with gr.Row():

        consultar = gr.Button(

            "🔎 Consultar",

            variant="primary"

        )

        microfone_btn = gr.Button(

            "🎤 Perguntar com Microfone",

            variant="secondary"

        )

        gerar_audio_btn = gr.Button(

            "🔊 Gerar Áudio"

        )

        limpar_btn = gr.Button(

            "🗑️ Limpar"

        )


    # ========================================================
    # RESPOSTA
    # ========================================================

    gr.Markdown(
        "## 📝 Resposta"
    )


    resposta = gr.Markdown()


    # ========================================================
    # ÁUDIO
    # ========================================================

    gr.Markdown(
        "## 🔊 Áudio"
    )


    audio_output = gr.Audio(

        label="Resposta em áudio",

        type="filepath",

        autoplay=False

    )


    # ========================================================
    # FONTES
    # ========================================================

    gr.Markdown(
        "## 📚 Fontes"
    )


    fontes = gr.Code(

        label="Documentos utilizados",

        language="json"

    )


    # ========================================================
    # EVENTOS
    # ========================================================


    consultar.click(

        fn=executar_rag,

        inputs=[pergunta],

        outputs=[

            resposta,

            fontes,

            audio_output

        ]

    )


    pergunta.submit(

        fn=executar_rag,

        inputs=[pergunta],

        outputs=[

            resposta,

            fontes,

            audio_output

        ]

    )

    microfone_btn.click(
        fn=transcrever_audio_e_executar_rag,
        inputs=[],
        outputs=[
            pergunta,
            resposta,
            fontes,
            audio_output
        ]
    )


    gerar_audio_btn.click(

        fn=gerar_audio_da_resposta,

        inputs=[resposta],

        outputs=[audio_output]

    )


    limpar_btn.click(

        fn=limpar_campos,

        inputs=[],

        outputs=[

            pergunta,

            resposta,

            audio_output,

            fontes

        ]

    )

FASE 31 — Executar Gradio

In [41]:
demo.launch(

    server_name="0.0.0.0",

    server_port=7869,

    share=False

)

* Running on local URL:  http://0.0.0.0:7869
* To create a public link, set `share=True` in `launch()`.


FASE 32 — Consulta diretamente no PostgreSQL

In [42]:
sql = f"""

SELECT

    id,

    nome_arquivo,

    pagina,

    chunk_numero,

    LEFT(
        content,
        300
    ) AS conteudo,

    data_ingestao

FROM {CONTENT_TABLE}

ORDER BY id DESC

LIMIT 50;

"""


with get_db_connection() as conn:

    df = pd.read_sql(
        sql,
        conn
    )


df

C:\Users\FABIO\AppData\Local\Temp\ipykernel_28064\1599470689.py:31: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
0,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
1,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
2,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
3,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
4,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
5,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
6,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
7,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
8,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao
9,id,nome_arquivo,pagina,chunk_numero,conteudo,data_ingestao


FASE 33 — Estatísticas finais

In [43]:
sql = f"""

SELECT

    COUNT(*) AS total_chunks,

    COUNT(
        DISTINCT documento_hash
    ) AS total_documentos,

    COUNT(
        DISTINCT nome_arquivo
    ) AS total_arquivos,

    COUNT(
        DISTINCT pagina
    ) AS total_paginas,

    MIN(data_ingestao)
        AS primeira_ingestao,

    MAX(data_ingestao)
        AS ultima_ingestao

FROM {CONTENT_TABLE};

"""


with get_db_connection() as conn:

    df = pd.read_sql(
        sql,
        conn
    )


df

C:\Users\FABIO\AppData\Local\Temp\ipykernel_28064\2991380437.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


,total_chunks,total_documentos,total_arquivos,total_paginas,primeira_ingestao,ultima_ingestao
0,total_chunks,total_documentos,total_arquivos,total_paginas,primeira_ingestao,ultima_ingestao


FASE 34 — Fluxo definitivo da aplicação

A arquitetura deste notebook fica:

                     ┌────────────────────┐
                     │  DIRETÓRIO DE PDFs │
                     └──────────┬─────────┘
                                │
                                ▼
                         ┌──────────────┐
                         │   PyMuPDF    │
                         └──────┬───────┘
                                │
                                ▼
                         TEXTO DO PDF
                                │
                                ▼
                     RecursiveCharacter
                         TextSplitter
                                │
                                ▼
                         CHUNKS DE TEXTO
                                │
                                ▼
                       nomic-embed-text
                                │
                         VECTOR(768)
                                │
                                ▼
                  ┌─────────────────────────┐
                  │      PostgreSQL 18      │
                  │                         │
                  │      rag_conteudo       │
                  │                         │
                  │ id                      │
                  │ documento_hash          │
                  │ nome_arquivo            │
                  │ pagina                  │
                  │ chunk_numero            │
                  │ content                 │
                  │ embedding               │
                  │ metadata                │
                  │ data_ingestao           │
                  └────────────┬────────────┘
                               │
                               │
                         PERGUNTA
                               │
                               ▼
                       nomic-embed-text
                               │
                               ▼
                         pgvector
                               │
                               ▼
                      TOP_K documentos
                               │
                               ▼
                         CONTEXTO RAG
                               │
                               ▼
                          qwen3:8b
                               │
                               ▼
                       RESPOSTA ESCRITA
                               │
                               ▼
                    ┌─────────────────────┐
                    │ ÁUDIO FOI SOLICITADO?│
                    └──────────┬──────────┘
                               │
                       ┌───────┴───────┐
                      NÃO             SIM
                       │               │
                       │               ▼
                       │             gTTS
                       │               │
                       │               ▼
                       │              MP3
                       │               │
                       └───────┬───────┘
                               ▼
                            Gradio

O fluxo da aplicação com chunking semântico agora é:

```
                     ┌────────────────────┐
                     │  DIRETÓRIO DE PDFs │
                     └──────────┬─────────┘
                                │
                                ▼
                         ┌──────────────┐
                         │   PyMuPDF    │
                         └──────┬───────┘
                                │
                                ▼
                         TEXTO DO PDF
                                │
                                ▼
                      NLTK Sentence Tokenizer
                                │
                                ▼
                       Sentenças de Texto
                                │
                                ▼
                       nomic-embed-text
                                │
                                ▼
                         Embeddings de Sentenças
                                │
                                ▼
                    Chunking Semântico (threshold)
                                │
                                ▼
                         CHUNKS DE TEXTO
                                │
                                ▼
                       nomic-embed-text
                                │
                         VECTOR(768)
                                │
                                ▼
                  ┌─────────────────────────┐
                  │      PostgreSQL 18      │
                  │                         │
                  │      rag_conteudo       │
                  │                         │
                  │ id                      │
                  │ documento_hash          │
                  │ nome_arquivo            │
                  │ pagina                  │
                  │ chunk_numero            │
                  │ content                 │
                  │ embedding               │
                  │ metadata                │
                  │ data_ingestao           │
                  └────────────┬────────────┘
                               │
                               │
                         PERGUNTA
                               │
                               ▼
                       nomic-embed-text
                               │
                               ▼
                         pgvector
                               │
                               ▼
                      TOP_K documentos
                               │
                               ▼
                         CONTEXTO RAG
                               │
                               ▼
                          qwen3:8b
                               │
                               ▼
                       RESPOSTA ESCRITA
                               │
                               ▼
                    ┌─────────────────────┐
                    │ ÁUDIO FOI SOLICITADO?│
                    └──────────┬──────────┘
                               │
                       ┌───────┴───────┐
                      NÃO             SIM
                       │               │
                       │               ▼
                       │             gTTS
                       │               │
                       │               ▼
                       │              MP3
                       │               │
                       └───────┬───────┘
                               ▼
                            Gradio
```